In [ ]:
# 1-dataset model (HTCas9)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_LbCas12aRVRR():
    """
    Loads (60,4) data from 'Feature_guide_baseonly_PAM_complete_LbCas12aRVRR_TTTV_filtered_reduced_seq_feature.txt'.
    """
    data_branch1 = []
    current_array = []
    with open('Feature_guide_baseonly_PAM_complete_LbCas12aRVRR_TTTV_filtered_reduced_seq_feature.txt', 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_LbCas12aRVRR():
    with open('LbCas12aRVRR_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # Rebuild for final 100-trial evaluation by loading the entire saved model
    model = torch.load(f"trial_{0}_model_HTCas9_combined_unique_CNN1_only_reduced_feature.pt", weights_only=False)
    model.eval()
    
    # final hold-out evaluation
    # load data

    X1_LbCas12aRVRR = load_branch1_data_LbCas12aRVRR()
    X1_LbCas12aRVRR   = np.asarray(X1_LbCas12aRVRR)
    X1 = np.concatenate([X1_LbCas12aRVRR], axis=0) 

    rates_LbCas12aRVRR = load_reaction_rates_LbCas12aRVRR()
    rates_LbCas12aRVRR   = np.asarray(rates_LbCas12aRVRR)
    rates = np.concatenate([rates_LbCas12aRVRR], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen LbCas12aRVRR dataset
    np.random.seed(42)
    full_indices_LbCas12aRVRR = np.arange(len(rates_LbCas12aRVRR))
    selected_indices_LbCas12aRVRR = np.random.choice(len(full_indices_LbCas12aRVRR), size=len(full_indices_LbCas12aRVRR), replace=False)
    unseen_indices_LbCas12aRVRR = np.setdiff1d(full_indices_LbCas12aRVRR, selected_indices_LbCas12aRVRR)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_LbCas12aRVRR = Subset(hybrid_dataset, unseen_indices_LbCas12aRVRR)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_LbCas12aRVRR = Subset(hybrid_dataset, selected_indices_LbCas12aRVRR)
    trial_loader = DataLoader(selected_set_LbCas12aRVRR, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    preds_trial = []
    labels_trial = []
    with torch.no_grad():
        for xx1_b, yy_b in trial_loader:
            p = model(xx1_b)
            preds_trial.append(p.item())
            labels_trial.append(yy_b.item())
        sp_corr, _ = spearmanr(labels_trial, preds_trial)
        print(sp_corr)
        print(f"Average Spearman Correlation for the selected dataset: {sp_corr}")

    with open('HTCas9_combined_preds_LbCas12aRVRR_sequence_only.txt', 'w') as f:
        for val in preds_trial:
            f.write(f"{val}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
0.3767251588441338
Average Spearman Correlation for the selected dataset: 0.3767251588441338


In [ ]:
# 2-dataset model (HTCas9+HT11)

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_LbCas12aRVRR():
    """
    Loads (60,4) data from 'Feature_guide_baseonly_PAM_complete_LbCas12aRVRR_TTTV_filtered_reduced_seq_feature.txt'.
    """
    data_branch1 = []
    current_array = []
    with open('Feature_guide_baseonly_PAM_complete_LbCas12aRVRR_TTTV_filtered_reduced_seq_feature.txt', 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_LbCas12aRVRR():
    with open('LbCas12aRVRR_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # Rebuild for final 100-trial evaluation by loading the entire saved model
    model = torch.load(f"trial_{7}_model_HTCas9_HT11_combined_unique_CNN1_only_reduced_feature.pt", weights_only=False)
    model.eval()
    
    # final hold-out evaluation
    # load data

    X1_LbCas12aRVRR = load_branch1_data_LbCas12aRVRR()
    X1_LbCas12aRVRR   = np.asarray(X1_LbCas12aRVRR)
    X1 = np.concatenate([X1_LbCas12aRVRR], axis=0) 

    rates_LbCas12aRVRR = load_reaction_rates_LbCas12aRVRR()
    rates_LbCas12aRVRR   = np.asarray(rates_LbCas12aRVRR)
    rates = np.concatenate([rates_LbCas12aRVRR], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen LbCas12aRVRR dataset
    np.random.seed(42)
    full_indices_LbCas12aRVRR = np.arange(len(rates_LbCas12aRVRR))
    selected_indices_LbCas12aRVRR = np.random.choice(len(full_indices_LbCas12aRVRR), size=len(full_indices_LbCas12aRVRR), replace=False)
    unseen_indices_LbCas12aRVRR = np.setdiff1d(full_indices_LbCas12aRVRR, selected_indices_LbCas12aRVRR)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_LbCas12aRVRR = Subset(hybrid_dataset, unseen_indices_LbCas12aRVRR)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_LbCas12aRVRR = Subset(hybrid_dataset, selected_indices_LbCas12aRVRR)
    trial_loader = DataLoader(selected_set_LbCas12aRVRR, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    preds_trial = []
    labels_trial = []
    with torch.no_grad():
        for xx1_b, yy_b in trial_loader:
            p = model(xx1_b)
            preds_trial.append(p.item())
            labels_trial.append(yy_b.item())
        sp_corr, _ = spearmanr(labels_trial, preds_trial)
        print(sp_corr)
        print(f"Average Spearman Correlation for the selected dataset: {sp_corr}")

    with open('HTCas9_HT11_combined_preds_LbCas12aRVRR_sequence_only.txt', 'w') as f:
        for val in preds_trial:
            f.write(f"{val}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
0.439558305743249
Average Spearman Correlation for the selected dataset: 0.439558305743249


In [ ]:
# 4-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5))

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_LbCas12aRVRR():
    """
    Loads (60,4) data from 'Feature_guide_baseonly_PAM_complete_LbCas12aRVRR_TTTV_filtered_reduced_seq_feature.txt'.
    """
    data_branch1 = []
    current_array = []
    with open('Feature_guide_baseonly_PAM_complete_LbCas12aRVRR_TTTV_filtered_reduced_seq_feature.txt', 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_LbCas12aRVRR():
    with open('LbCas12aRVRR_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # Rebuild for final 100-trial evaluation by loading the entire saved model
    model = torch.load(f"trial_{3}_model_HTCas9_HT11_RfxCas13d_combined_unique_CNN1_only_reduced_feature.pt", weights_only=False)
    model.eval()
    
    # final hold-out evaluation
    # load data

    X1_LbCas12aRVRR = load_branch1_data_LbCas12aRVRR()
    X1_LbCas12aRVRR   = np.asarray(X1_LbCas12aRVRR)
    X1 = np.concatenate([X1_LbCas12aRVRR], axis=0) 

    rates_LbCas12aRVRR = load_reaction_rates_LbCas12aRVRR()
    rates_LbCas12aRVRR   = np.asarray(rates_LbCas12aRVRR)
    rates = np.concatenate([rates_LbCas12aRVRR], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen LbCas12aRVRR dataset
    np.random.seed(42)
    full_indices_LbCas12aRVRR = np.arange(len(rates_LbCas12aRVRR))
    selected_indices_LbCas12aRVRR = np.random.choice(len(full_indices_LbCas12aRVRR), size=len(full_indices_LbCas12aRVRR), replace=False)
    unseen_indices_LbCas12aRVRR = np.setdiff1d(full_indices_LbCas12aRVRR, selected_indices_LbCas12aRVRR)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_LbCas12aRVRR = Subset(hybrid_dataset, unseen_indices_LbCas12aRVRR)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_LbCas12aRVRR = Subset(hybrid_dataset, selected_indices_LbCas12aRVRR)
    trial_loader = DataLoader(selected_set_LbCas12aRVRR, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    preds_trial = []
    labels_trial = []
    with torch.no_grad():
        for xx1_b, yy_b in trial_loader:
            p = model(xx1_b)
            preds_trial.append(p.item())
            labels_trial.append(yy_b.item())
        sp_corr, _ = spearmanr(labels_trial, preds_trial)
        print(sp_corr)
        print(f"Average Spearman Correlation for the selected dataset: {sp_corr}")

    with open('HTCas9_HT11_RfxCas13d_combined_preds_LbCas12aRVRR_sequence_only.txt', 'w') as f:
        for val in preds_trial:
            f.write(f"{val}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
0.4253569202194604
Average Spearman Correlation for the selected dataset: 0.4253569202194604


In [ ]:
# 5-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq)

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_LbCas12aRVRR():
    """
    Loads (60,4) data from 'Feature_guide_baseonly_PAM_complete_LbCas12aRVRR_TTTV_filtered_reduced_seq_feature.txt'.
    """
    data_branch1 = []
    current_array = []
    with open('Feature_guide_baseonly_PAM_complete_LbCas12aRVRR_TTTV_filtered_reduced_seq_feature.txt', 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_LbCas12aRVRR():
    with open('LbCas12aRVRR_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # Rebuild for final 100-trial evaluation by loading the entire saved model
    model = torch.load(f"trial_{0}_model_HTCas9_HT11_RfxCas13d_change_seq_sampled_2_combined_unique_CNN1_only_reduced_feature.pt", weights_only=False)
    model.eval()
    
    # final hold-out evaluation
    # load data

    X1_LbCas12aRVRR = load_branch1_data_LbCas12aRVRR()
    X1_LbCas12aRVRR   = np.asarray(X1_LbCas12aRVRR)
    X1 = np.concatenate([X1_LbCas12aRVRR], axis=0) 

    rates_LbCas12aRVRR = load_reaction_rates_LbCas12aRVRR()
    rates_LbCas12aRVRR   = np.asarray(rates_LbCas12aRVRR)
    rates = np.concatenate([rates_LbCas12aRVRR], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen LbCas12aRVRR dataset
    np.random.seed(42)
    full_indices_LbCas12aRVRR = np.arange(len(rates_LbCas12aRVRR))
    selected_indices_LbCas12aRVRR = np.random.choice(len(full_indices_LbCas12aRVRR), size=len(full_indices_LbCas12aRVRR), replace=False)
    unseen_indices_LbCas12aRVRR = np.setdiff1d(full_indices_LbCas12aRVRR, selected_indices_LbCas12aRVRR)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_LbCas12aRVRR = Subset(hybrid_dataset, unseen_indices_LbCas12aRVRR)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_LbCas12aRVRR = Subset(hybrid_dataset, selected_indices_LbCas12aRVRR)
    trial_loader = DataLoader(selected_set_LbCas12aRVRR, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    preds_trial = []
    labels_trial = []
    with torch.no_grad():
        for xx1_b, yy_b in trial_loader:
            p = model(xx1_b)
            preds_trial.append(p.item())
            labels_trial.append(yy_b.item())
        sp_corr, _ = spearmanr(labels_trial, preds_trial)
        print(sp_corr)
        print(f"Average Spearman Correlation for the selected dataset: {sp_corr}")

    with open('HTCas9_HT11_RfxCas13d_change_seq_combined_preds_LbCas12aRVRR_sequence_only.txt', 'w') as f:
        for val in preds_trial:
            f.write(f"{val}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
0.4260608354968791
Average Spearman Correlation for the selected dataset: 0.4260608354968791


In [ ]:
# 6-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq+TIGER)

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_LbCas12aRVRR():
    """
    Loads (60,4) data from 'Feature_guide_baseonly_PAM_complete_LbCas12aRVRR_TTTV_filtered_reduced_seq_feature.txt'.
    """
    data_branch1 = []
    current_array = []
    with open('Feature_guide_baseonly_PAM_complete_LbCas12aRVRR_TTTV_filtered_reduced_seq_feature.txt', 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_LbCas12aRVRR():
    with open('LbCas12aRVRR_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # Rebuild for final 100-trial evaluation by loading the entire saved model
    model = torch.load(f"trial_{6}_model_HTCas9_HT11_RfxCas13d_change_seq_sampled_2_tiger_combined_unique_CNN1_only_reduced_feature.pt", weights_only=False)
    model.eval()
    
    # final hold-out evaluation
    # load data

    X1_LbCas12aRVRR = load_branch1_data_LbCas12aRVRR()
    X1_LbCas12aRVRR   = np.asarray(X1_LbCas12aRVRR)
    X1 = np.concatenate([X1_LbCas12aRVRR], axis=0) 

    rates_LbCas12aRVRR = load_reaction_rates_LbCas12aRVRR()
    rates_LbCas12aRVRR   = np.asarray(rates_LbCas12aRVRR)
    rates = np.concatenate([rates_LbCas12aRVRR], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen LbCas12aRVRR dataset
    np.random.seed(42)
    full_indices_LbCas12aRVRR = np.arange(len(rates_LbCas12aRVRR))
    selected_indices_LbCas12aRVRR = np.random.choice(len(full_indices_LbCas12aRVRR), size=len(full_indices_LbCas12aRVRR), replace=False)
    unseen_indices_LbCas12aRVRR = np.setdiff1d(full_indices_LbCas12aRVRR, selected_indices_LbCas12aRVRR)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_LbCas12aRVRR = Subset(hybrid_dataset, unseen_indices_LbCas12aRVRR)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_LbCas12aRVRR = Subset(hybrid_dataset, selected_indices_LbCas12aRVRR)
    trial_loader = DataLoader(selected_set_LbCas12aRVRR, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    preds_trial = []
    labels_trial = []
    with torch.no_grad():
        for xx1_b, yy_b in trial_loader:
            p = model(xx1_b)
            preds_trial.append(p.item())
            labels_trial.append(yy_b.item())
        sp_corr, _ = spearmanr(labels_trial, preds_trial)
        print(sp_corr)
        print(f"Average Spearman Correlation for the selected dataset: {sp_corr}")

    with open('HTCas9_HT11_RfxCas13d_change_seq_sampled_2_tiger_combined_preds_LbCas12aRVRR_sequence_only.txt', 'w') as f:
        for val in preds_trial:
            f.write(f"{val}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
0.42297275273439544
Average Spearman Correlation for the selected dataset: 0.42297275273439544


In [ ]:
# 7-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq+TIGER+iMeta)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_LbCas12aRVRR():
    """
    Loads (60,4) data from 'Feature_guide_baseonly_PAM_complete_LbCas12aRVRR_TTTV_filtered_reduced_seq_feature.txt'.
    """
    data_branch1 = []
    current_array = []
    with open('Feature_guide_baseonly_PAM_complete_LbCas12aRVRR_TTTV_filtered_reduced_seq_feature.txt', 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_LbCas12aRVRR():
    with open('LbCas12aRVRR_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # Rebuild for final 100-trial evaluation by loading the entire saved model
    model = torch.load(f"trial_{7}_model_HTCas9_HT11_RfxCas13d_change_seq_sampled_2_tiger_iMeta_combined_unique_CNN1_only_reduced_feature.pt", weights_only=False)
    model.eval()
    
    # final hold-out evaluation
    # load data

    X1_LbCas12aRVRR = load_branch1_data_LbCas12aRVRR()
    X1_LbCas12aRVRR   = np.asarray(X1_LbCas12aRVRR)
    X1 = np.concatenate([X1_LbCas12aRVRR], axis=0) 

    rates_LbCas12aRVRR = load_reaction_rates_LbCas12aRVRR()
    rates_LbCas12aRVRR   = np.asarray(rates_LbCas12aRVRR)
    rates = np.concatenate([rates_LbCas12aRVRR], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen LbCas12aRVRR dataset
    np.random.seed(42)
    full_indices_LbCas12aRVRR = np.arange(len(rates_LbCas12aRVRR))
    selected_indices_LbCas12aRVRR = np.random.choice(len(full_indices_LbCas12aRVRR), size=len(full_indices_LbCas12aRVRR), replace=False)
    unseen_indices_LbCas12aRVRR = np.setdiff1d(full_indices_LbCas12aRVRR, selected_indices_LbCas12aRVRR)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_LbCas12aRVRR = Subset(hybrid_dataset, unseen_indices_LbCas12aRVRR)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_LbCas12aRVRR = Subset(hybrid_dataset, selected_indices_LbCas12aRVRR)
    trial_loader = DataLoader(selected_set_LbCas12aRVRR, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    preds_trial = []
    labels_trial = []
    with torch.no_grad():
        for xx1_b, yy_b in trial_loader:
            p = model(xx1_b)
            preds_trial.append(p.item())
            labels_trial.append(yy_b.item())
        sp_corr, _ = spearmanr(labels_trial, preds_trial)
        print(sp_corr)
        print(f"Average Spearman Correlation for the selected dataset: {sp_corr}")

    with open('HTCas9_HT11_RfxCas13d_change_seq_sampled_2_tiger_iMeta_combined_preds_LbCas12aRVRR_sequence_only.txt', 'w') as f:
        for val in preds_trial:
            f.write(f"{val}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
0.41256374415985886
Average Spearman Correlation for the selected dataset: 0.41256374415985886
